# Cross-Selling Term Deposits with Spark ML and Structured Streaming

A bank calls existing customers to sell a term deposit; only about 11% say yes. This notebook builds the whole workflow in PySpark (local mode): Spark SQL exploration, a feature pipeline, five classifiers with class weighting, cross-validated tuning, and finally a **Structured Streaming** job that scores customers as files arrive and is checked row by row against batch scoring.

**Leakage note.** The dataset's `duration` column is the length of the sales call, known only after the call ends and almost determines the outcome. It is excluded, so the models only use what the bank knows before dialling.

**Data:** [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), UCI Machine Learning Repository, CC BY 4.0 (`bank-additional-full.csv`, 41,188 calls, Portugal, 2008-2010). Moro, S., Cortez, P. and Rita, P. (2014), *Decision Support Systems* 62.

In [ ]:
import io
import shutil
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.classification import (DecisionTreeClassifier, GBTClassifier, LogisticRegression,
                                       MultilayerPerceptronClassifier, RandomForestClassifier)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, functions as F

SEED = 2022
spark = (SparkSession.builder.master("local[2]").appName("bank-marketing")
         .config("spark.sql.shuffle.partitions", "8").config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.version

## Data

In [ ]:
path = Path("data/bank-additional-full.csv")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urlretrieve("https://archive.ics.uci.edu/static/public/222/bank+marketing.zip", "data/bank-marketing.zip")
    inner = zipfile.ZipFile("data/bank-marketing.zip").read("bank-additional.zip")
    path.write_bytes(zipfile.ZipFile(io.BytesIO(inner)).read("bank-additional/bank-additional-full.csv"))

raw = spark.read.csv(str(path), header=True, sep=";", inferSchema=True)
df = (raw.toDF(*[c.replace(".", "_") for c in raw.columns])
         .withColumn("label", (F.col("y") == "yes").cast("double"))
         .drop("y", "duration"))
df.cache()
print(df.count(), "rows,", len(df.columns), "columns")
df.printSchema()

## Exploration with Spark SQL

In [ ]:
df.createOrReplaceTempView("calls")
spark.sql("""
    SELECT ROUND(AVG(label), 4) AS yes_rate, COUNT(*) AS calls FROM calls
""").show()
spark.sql("""
    SELECT poutcome, COUNT(*) AS calls, ROUND(AVG(label), 3) AS yes_rate
    FROM calls GROUP BY poutcome ORDER BY yes_rate DESC
""").show()
spark.sql("""
    SELECT month, COUNT(*) AS calls, ROUND(AVG(label), 3) AS yes_rate
    FROM calls GROUP BY month ORDER BY yes_rate DESC
""").show()

In [ ]:
by_rate = (df.groupBy(F.round("euribor3m").alias("euribor3m")).agg(F.avg("label").alias("yes_rate"), F.count("*").alias("calls"))
             .orderBy("euribor3m").toPandas())
ax = by_rate.plot(x="euribor3m", y="yes_rate", marker="o", legend=False, figsize=(8, 4),
                  title="Yes rate by 3-month Euribor (rounded)")
ax.set_ylabel("yes rate")
plt.show()

Previous campaign success and the macro environment (low Euribor, in 2009-2010) are by far the strongest signals; the month mostly proxies for that environment.

## Feature pipeline and split

Categoricals are indexed and one-hot encoded, numerics assembled and standardised, all inside one `Pipeline` fitted on the training split. Positives get a class weight so that both classes count equally in the loss.

In [ ]:
categorical = [c for c, t in df.dtypes if t == "string"]
numeric = [c for c, t in df.dtypes if t != "string" and c != "label"]

train, test = df.randomSplit([0.7, 0.3], seed=SEED)
pos_rate = train.agg(F.avg("label")).first()[0]
weight = F.when(F.col("label") == 1, 0.5 / pos_rate).otherwise(0.5 / (1 - pos_rate))
train, test = train.withColumn("weight", weight).cache(), test.withColumn("weight", F.lit(1.0)).cache()
print(f"train {train.count()} / test {test.count()}, positive rate {pos_rate:.3f}")

features = [
    StringIndexer(inputCols=categorical, outputCols=[c + "_idx" for c in categorical], handleInvalid="keep"),
    OneHotEncoder(inputCols=[c + "_idx" for c in categorical], outputCols=[c + "_ohe" for c in categorical]),
    VectorAssembler(inputCols=numeric + [c + "_ohe" for c in categorical], outputCol="raw_features"),
    StandardScaler(inputCol="raw_features", outputCol="features", withMean=True),
]
n_features = Pipeline(stages=features).fit(train).transform(train.limit(1)).first()["features"].size
n_features

## Models

In [ ]:
roc = BinaryClassificationEvaluator(metricName="areaUnderROC")
pr = BinaryClassificationEvaluator(metricName="areaUnderPR")

def report(name, model):
    pred = model.transform(test).withColumn("p", vector_to_array("probability")[1])
    counts = pred.groupBy("label", "prediction").count().toPandas().set_index(["label", "prediction"])["count"]
    tp, fp = counts.get((1.0, 1.0), 0), counts.get((0.0, 1.0), 0)
    fn = counts.get((1.0, 0.0), 0)
    return {"ROC AUC": roc.evaluate(pred), "PR AUC": pr.evaluate(pred),
            "Recall": tp / (tp + fn), "Precision": tp / max(tp + fp, 1)}

lr = LogisticRegression(weightCol="weight", maxIter=100)
grid = ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.01, 0.1]).addGrid(lr.elasticNetParam, [0.0, 0.5]).build()
cv_lr = CrossValidator(estimator=Pipeline(stages=features + [lr]), estimatorParamMaps=grid,
                       evaluator=pr, numFolds=3, seed=SEED, parallelism=2)

gbt = GBTClassifier(weightCol="weight", seed=SEED)
grid_gbt = ParamGridBuilder().addGrid(gbt.maxDepth, [3, 5]).addGrid(gbt.maxIter, [50, 100]).build()
cv_gbt = CrossValidator(estimator=Pipeline(stages=features + [gbt]), estimatorParamMaps=grid_gbt,
                        evaluator=pr, numFolds=3, seed=SEED, parallelism=2)

candidates = {
    "Logistic regression (CV-tuned)": cv_lr,
    "Decision tree": Pipeline(stages=features + [DecisionTreeClassifier(weightCol="weight", maxDepth=6, seed=SEED)]),
    "Random forest": Pipeline(stages=features + [RandomForestClassifier(weightCol="weight", numTrees=200, maxDepth=8, seed=SEED)]),
    "Gradient-boosted trees (CV-tuned)": cv_gbt,
    "Multilayer perceptron": Pipeline(stages=features + [MultilayerPerceptronClassifier(layers=[n_features, 32, 16, 2], maxIter=200, seed=SEED)]),
}
fitted, rows = {}, {}
for name, est in candidates.items():
    fitted[name] = est.fit(train)
    rows[name] = report(name, fitted[name])
results = pd.DataFrame(rows).T.sort_values("PR AUC", ascending=False)
results.round(3)

In [ ]:
best_gbt = fitted["Gradient-boosted trees (CV-tuned)"].bestModel
best_lr = fitted["Logistic regression (CV-tuned)"].bestModel.stages[-1]
print("GBT:", {p.name: v for p, v in best_gbt.stages[-1].extractParamMap().items() if p.name in ("maxDepth", "maxIter")})
print("LR: regParam", best_lr.getRegParam(), "elasticNetParam", best_lr.getElasticNetParam())

The multilayer perceptron in Spark ML has no sample weights, so it is trained on the imbalanced data as is; its recall shows the cost of that.

## Structured Streaming: scoring customers as they arrive

The test set is written as ten CSV files. A streaming query reads the folder with a fixed schema, applies the fitted gradient-boosting pipeline and writes scores to a sink; the result is then compared, customer by customer, with batch scoring of the same rows.

In [ ]:
model = best_gbt
incoming = Path("stream/incoming")
shutil.rmtree("stream", ignore_errors=True)
incoming.mkdir(parents=True)
(test.drop("weight").withColumn("customer_id", F.monotonically_increasing_id())
     .repartition(10).write.csv(str(incoming), header=True, mode="overwrite"))
schema = spark.read.csv(str(incoming), header=True, inferSchema=True).schema

stream = spark.readStream.schema(schema).option("header", True).option("maxFilesPerTrigger", 2).csv(str(incoming))
scores = model.transform(stream).select("customer_id", "label", vector_to_array("probability")[1].alias("p"))
query = (scores.writeStream.format("parquet").option("path", "stream/scores")
         .option("checkpointLocation", "stream/checkpoint").trigger(availableNow=True).start())
query.awaitTermination()

streamed = spark.read.parquet("stream/scores")
batch = model.transform(spark.read.schema(schema).csv(str(incoming), header=True)) \
             .select("customer_id", vector_to_array("probability")[1].alias("p_batch"))
check = streamed.join(batch, "customer_id")
n_stream, n_batch = streamed.count(), batch.count()
max_diff = check.agg(F.max(F.abs(F.col("p") - F.col("p_batch")))).first()[0]
print(f"streamed {n_stream} rows in {len(query.recentProgress)} micro-batches; batch {n_batch}; max |difference| = {max_diff}")

In [ ]:
# Checks: streaming matched batch exactly, and the best model clearly beats the base rate.
assert n_stream == n_batch == check.count()
assert max_diff == 0.0
assert results["PR AUC"].max() > 2 * pos_rate

## Conclusion

CONCLUSION_PLACEHOLDER

In [ ]:
spark.stop()